## Imports en MNIST laden

De MNIST-dataset bevat 70.000 handgeschreven cijfers (0-9).
Keras downloadt de data automatisch de eerste keer (~11 MB).

In [ ]:
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print('=== Dataset geladen ===')
print('Trainingsset afbeeldingen :', X_train.shape)
print('Trainingsset labels       :', y_train.shape)
print('Testset afbeeldingen      :', X_test.shape)
print('Testset labels            :', y_test.shape)
print('Pixelwaarden: min =', X_train.min(), ', max =', X_train.max())
print('Unieke klassen:', np.unique(y_train))

# **P3 Small Decision Tree**

### **Opdracht: Ontwerp je Eigen Features & Bouw een Simpele Decision Tree**
Je gaat zelf ontdekken hoe je informatie uit afbeeldingen kunt halen door:

je mag geen externa libraries gebruiken (behalve numpy)
Zelf features te bedenken (kenmerken)
Die features meetbaar te maken met Python
Een héél simpele decision tree te bouwen die op basis van jouw features probeert een getal te herkennen
Te onderzoeken waarom bepaalde features wel of niet goed werken
Je hoeft hiervoor nog niets te weten over machine learning of accuracy. Deze opdracht gaat om begrijpen, onderzoeken, uitleggen, en experimenteren.

### **Deel 1 Maak losse methodes per feature**
Een feature is één getal dat iets zegt over de afbeelding. Voorbeelden (je mag kiezen of totaal iets anders verzinnen):

Hoeveel donkere pixels zitten er in de afbeelding?
Is de afbeelding links donkerder dan rechts?
Hoe hoog en hoe breed is de “vorm”?
Hoe symmetrisch is het cijfer?
iets met stdev?
Hoeveel aparte stukjes donkere pixels bevat het?
Kies 5 of meer features die volgens jou iets kunnen zeggen over het cijfer.

1. Welke features heb je gekozen en waarom?

    - Percentage donkere pixels
    - Links vs rechts donkerheid
    - Boven vs onder donkerheid
    - Symmetrie links en rechts
    - Hoeveelheid zwarte pixels in het midden 


2. Leg de werking uit

    Bekijk code

In [ ]:
import numpy as np

# 1. Percentage donkere pixels
def procent_donkere_pixels(img):
    totaal_pixels = img.size
    donkere_pixels = np.sum(img < 128)  
    percentage = (donkere_pixels / totaal_pixels) * 100

    if percentage < 10:       
        return 1
    elif percentage > 30:    
        return 8
    else: 
        return 0

# 2. Donkerheid links vs rechts
def links_rechts_donkerheid(img):
    h, b = img.shape
    links = img[:, :b//2]
    rechts = img[:, b//2:]

    donkerheid_links = np.sum(links < 128)
    donkerheid_rechts = np.sum(rechts < 128)

    if donkerheid_links > donkerheid_rechts + 20:
        return 5
    elif donkerheid_rechts > donkerheid_links + 20:
        return 3
    else:
        return 0

# 3. Donkerheid boven vs onder
def boven_onder_donkerheid(img):
    h, b = img.shape
    boven = img[:h//2, :]
    onder = img[h//2:, :]

    donkerheid_boven = np.sum(boven < 128)
    donkerheid_onder = np.sum(onder < 128)

    if donkerheid_boven > donkerheid_onder + 20:
        return 7       
    elif donkerheid_onder > donkerheid_boven + 20:
        return 4      
    else:
        return 0

# 4. Symmetrie links-rechts
def symmetrie_links_rechts(img):
    h, b = img.shape
    links = img[:, :b//2]
    rechts = img[:, b//2:]

    verschil = np.mean(np.abs(links - np.fliplr(rechts)))

    if verschil < 40:
        return 0 
    else:
        return 3

# 5. Hoeveelheid zwarte pixels in het midden
def zwarte_pixels_midden(img):
    h, b = img.shape
    midden = img[h//4:3*h//4, b//4:3*b//4]
    zwarte_pixels = np.sum(midden < 128)

    if zwarte_pixels > 60:
        return 8
    else:
        return 0
    
# Test met eerste afbeelding uit de dataset
img = X_train[0]

print("Echt cijfer          :", y_train[0])
print("Procent donkere pixels:", procent_donkere_pixels(img))
print("Links rechts          :", links_rechts_donkerheid(img))
print("Boven onder           :", boven_onder_donkerheid(img))
print("Symmetrie             :", symmetrie_links_rechts(img))
print("Pixels midden         :", zwarte_pixels_midden(img))

### **Deel 2: Combineer je features**
Maak een methode die per afbeelding een matrix van deze features aanmaakt. Roep deze functie aan voor een (sub)set aan Mnist afbeeldingen.

Je mag zelf bepalen in welke datastructuur je het resultaat opslaat. Het is in ieder geval belangrijk dat je de afbeelding en de bijbehorende features opslaat.

In [ ]:
def extract_features(img):
    return [
        procent_donkere_pixels(img),
        links_rechts_donkerheid(img),
        boven_onder_donkerheid(img),
        symmetrie_links_rechts(img),
        zwarte_pixels_midden(img)
    ]

def create_feature_matrix(images):
    dataset =[]
    for idx, img in enumerate(images):
        features = extract_features(img)
        dataset.append((idx,features))
    return dataset

image_subset = X_train[:1000]

feature_dataset = create_feature_matrix(image_subset)


1. Op hoeveel afbeeldingen heb je extract_features functie toegepast? Waarom heb je voor dit aantal gekozen?

    1.000 afbeeldingen, dit is voldoende om de feeatures te testen, en is sneller te verwerken dan alle 60.000 afbeeldingen van MNIST

2. Welke datastructuur heb je gebruikt om je nieuwe dataset in op te slaan en waarom?

    List, want je kunt er makkelijk doorheen en het houd de volgorde van de afbeelding bij.

### **Deel 3: Decision tree maken**
We gebruiken geen scikit-learn . Je bouwt een klein boompje met splitsingen.

Je tree werkt zo:

“Als feature X kleiner is dan drempel T → linkerkind Anders → rechterkind”

In [ ]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def predict_tree(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature_index] < node.threshold:
        return predict_tree(node.left, x)
    else:
        return predict_tree(node.right, x)

leaf0 = Node(value=0)
leaf1 = Node(value=1)
root = Node(feature_index=0, threshold=0.5, left=leaf0, right=leaf1)

sample = feature_dataset[0][1]
print("Voorspelling:", predict_tree(root, sample))
print("Echt cijfer :", y_train[0])

### **Deel 4: Testen**
Je laat de boom op een aantal nieuwe afbeeldingen een voorspelling doen.

Bijvoorbeeld:

toon 10 nieuwe cijfers
laat de boom voorspellen
vergelijk met het echte label
bespreek wat opvalt

### **Deel 5: Runnen als embedded code**
Zoals aan het begin van de cursus besproken gaat het project over handgeschreven nummers herekennen op een MysteryDevice met de volgende eigenschappen:

Input scherm waarmee een nieuwe plaatje als ndarray aangemaakt kan worden door de gebruiker.
Zeer beperkt RAM (256 KB)
Beperkte opslag (1 MB voor programma + model)
Geen GPU
Embedded python
Kan je programma er al op runnen?